In [1]:
import torch
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from peft import LoraConfig, get_peft_model
import evaluate

/mnt/f/AsrFinetune/asr_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("changelinglab/easycall-dysarthria")

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [3]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-medium",
    language="italian",
    task="transcribe"
)

In [4]:
def normalize(text):
    return text.lower().strip()

def prepare(batch):
    audio_arrays = [x["array"] for x in batch["audio"]]

    batch["input_features"] = processor.feature_extractor(
        audio_arrays,
        sampling_rate=16000,
    ).input_features

    batch["labels"] = processor.tokenizer(
        [normalize(t) for t in batch["text"]],
        padding=True,
        truncation=True
    ).input_ids

    return batch

dataset = dataset.with_transform(prepare)

In [5]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")

model.config.use_cache = False
model.gradient_checkpointing_enable()

In [6]:
lora_config = LoraConfig(
    # r=16,
    r = 32,
    # lora_alpha=32,
    lora_alpha=64,
    # target_modules=["q_proj", "v_proj"],
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 18,874,368 || all params: 782,732,288 || trainable%: 2.4113


In [7]:
from dataclasses import dataclass
from typing import Any

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):

        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"]

        # manual shift
        decoder_input_ids = labels.clone()
        decoder_input_ids[:, 1:] = labels[:, :-1]
        decoder_input_ids[:, 0] = self.processor.tokenizer.bos_token_id

        labels = labels.masked_fill(
            labels == self.processor.tokenizer.pad_token_id,
            -100
        )

        batch["labels"] = labels
        batch["decoder_input_ids"] = decoder_input_ids

        return batch

In [8]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [normalize(s) for s in pred_str]
    label_str = [normalize(s) for s in label_str]

    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

In [ ]:
# training_args = Seq2SeqTrainingArguments(
#     output_dir="./whisper-medium-lora",

#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=16,

#     learning_rate=1e-4,
#     warmup_steps=100,

#     num_train_epochs=10,

#     fp16=True,

#     eval_strategy="epoch",
#     save_strategy="epoch",

#     predict_with_generate=False,  

#     dataloader_num_workers=0,
#     remove_unused_columns=False,

#     logging_steps=25,
#     save_total_limit=2,

#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss", 
# )
# training_args = Seq2SeqTrainingArguments(
#     output_dir="./whisper-medium-lora",

#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,

#     learning_rate=1e-4,
#     warmup_steps=100,

#     num_train_epochs=10,

#     fp16=True,

#     eval_strategy="no",
#     save_strategy="epoch",

#     predict_with_generate=False,

#     dataloader_num_workers=0,
#     remove_unused_columns=False,

#     logging_steps=25,
#     save_total_limit=2,

#     load_best_model_at_end=False,  
# )

In [12]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-medium-lora",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    warmup_steps=100,

    num_train_epochs=5,

    fp16=True,

    eval_strategy="no",

    save_strategy="epoch",
    save_total_limit=5,   

    predict_with_generate=False,

    dataloader_num_workers=0,
    remove_unused_columns=False,

    logging_steps=25,

    load_best_model_at_end=False,
)

In [ ]:
# from transformers import EarlyStoppingCallback

# data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     eval_dataset=dataset["validation"],
#     # tokenizer=processor.feature_extractor,
#     processing_class=processor.tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
#     callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
# )

In [15]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=processor.tokenizer,  
    data_collator=data_collator,
)

In [16]:
trainer.train()

/mnt/f/AsrFinetune/asr_env/lib/python3.9/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
25,0.188800
50,0.141500
75,0.145400
100,0.137000
125,0.131200
150,0.160300
175,0.130100
200,0.082800
225,0.134100
250,0.099100


/mnt/f/AsrFinetune/asr_env/lib/python3.9/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/mnt/f/AsrFinetune/asr_env/lib/python3.9/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/mnt/f/AsrFinetune/asr_env/lib/python3.9/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/mnt/f/AsrFinetune/asr_env/lib/python3.9/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


TrainOutput(global_step=3720, training_loss=0.05454656240600412, metrics={'train_runtime': 13476.1042, 'train_samples_per_second': 4.416, 'train_steps_per_second': 0.276, 'total_flos': 6.23483867676672e+19, 'train_loss': 0.05454656240600412, 'epoch': 5.0})

In [ ]:
import torch
import evaluate
from tqdm import tqdm

wer_metric = evaluate.load("wer")

def normalize(text):
    return text.lower().strip()

model.eval()
model.to("cuda")

predictions = []
references = []

for sample in tqdm(dataset["test"]):

    # prepare input
    input_features = processor.feature_extractor(
        sample["audio"]["array"],
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to("cuda")

    # generate prediction
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_length=64
        )

    # decode
    pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    ref_text = sample["text"]

    # normalize
    predictions.append(normalize(pred_text))
    references.append(normalize(ref_text))

# compute WER
wer = wer_metric.compute(predictions=predictions, references=references)

print(f"\n🔥 Test WER: {wer * 100:.2f}%")

In [17]:
import os
from transformers import WhisperForConditionalGeneration

checkpoints = sorted([
    os.path.join("./whisper-medium-lora", d)
    for d in os.listdir("./whisper-medium-lora")
    if d.startswith("checkpoint")
])

results = []

for ckpt in checkpoints:
    print(f"\nEvaluating {ckpt}")

    model = WhisperForConditionalGeneration.from_pretrained(ckpt).to("cuda")
    model.eval()

    preds = []
    refs = []

    for sample in dataset["validation"].select(range(500)):  # 🔥 subset for speed

        input_features = processor.feature_extractor(
            sample["audio"]["array"],
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to("cuda")

        with torch.no_grad():
            ids = model.generate(input_features, max_length=64)

        pred = processor.batch_decode(ids, skip_special_tokens=True)[0]
        ref = sample["text"]

        preds.append(pred.lower().strip())
        refs.append(ref.lower().strip())

    wer = wer_metric.compute(predictions=preds, references=refs)

    print(f"WER: {wer:.4f}")
    results.append((ckpt, wer))

# 🔥 sort best
results = sorted(results, key=lambda x: x[1])

print("\n🏆 Best checkpoints:")
for r in results:
    print(r)


Evaluating ./whisper-medium-lora/checkpoint-1488


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


WER: 0.4826

Evaluating ./whisper-medium-lora/checkpoint-2232
WER: 0.5495

Evaluating ./whisper-medium-lora/checkpoint-2976
WER: 0.6016

Evaluating ./whisper-medium-lora/checkpoint-3720
WER: 0.8904

Evaluating ./whisper-medium-lora/checkpoint-744
WER: 0.5374

🏆 Best checkpoints:
('./whisper-medium-lora/checkpoint-1488', 0.48262032085561496)
('./whisper-medium-lora/checkpoint-744', 0.5374331550802139)
('./whisper-medium-lora/checkpoint-2232', 0.5494652406417112)
('./whisper-medium-lora/checkpoint-2976', 0.6016042780748663)
('./whisper-medium-lora/checkpoint-3720', 0.8903743315508021)
